# Pandas — Basic to Advanced
**Practical Jupyter notebook | Beginner → Advanced**

Learn how to work with tabular data: create DataFrames, inspect and clean data, filter rows, group data, merge tables, handle dates, and prepare datasets for analysis and machine learning.

## 0. Setup

In [ ]:
# If needed, run in a notebook cell:
# %pip install pandas numpy openpyxl

import numpy as np
import pandas as pd

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 13. Series and DataFrame

In [ ]:
series = pd.Series([80, 92, 75], index=["Ali", "Sara", "Omar"], name="score")
print(series)
print("Sara's score:", series.loc["Sara"])

df = pd.DataFrame({
    "name": ["Ali", "Sara", "Omar"],
    "age": [22, 25, 23],
    "score": [80, 92, 75]
})
display(df)

**Core idea:** A `Series` is a labelled one-dimensional structure. A `DataFrame` is a table of labelled rows and columns.

## 14. Inspecting a DataFrame

In [ ]:
print(df.head())
print(df.tail(2))
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.info()
display(df.describe(numeric_only=True))

## 15. Selecting columns, rows, and cells

In [ ]:
print(df["name"])                 # one column -> Series
display(df[["name", "score"]])    # multiple columns -> DataFrame
display(df.loc[0])                # row by index label
display(df.iloc[0])               # row by integer position
print(df.loc[1, "score"])         # row label 1, column score
print(df.iloc[0, 2])              # first row, third column

### Filter rows

In [ ]:
display(df[df["score"] >= 80])
display(df[(df["age"] >= 23) & (df["score"] > 75)])
display(df[df["name"].isin(["Ali", "Omar"])])

## 16. Add, update, and remove columns

In [ ]:
df_work = df.copy()
df_work["passed"] = df_work["score"] >= 60
df_work["score_plus_5"] = df_work["score"] + 5
df_work.loc[df_work["name"] == "Omar", "score"] = 78
display(df_work)

without_age = df_work.drop(columns=["age"])
display(without_age)

## 17. Sorting and ranking

In [ ]:
display(df.sort_values("score", ascending=False))
df_ranked = df.assign(rank=df["score"].rank(method="dense", ascending=False))
display(df_ranked)

## 18. Missing values

In [ ]:
messy = pd.DataFrame({
    "name": ["Ali", "Sara", "Omar", "Mina"],
    "age": [22, np.nan, 23, np.nan],
    "score": [80, 92, np.nan, 70],
    "city": ["London", "Leeds", None, "London"]
})
display(messy)
print(messy.isna().sum())
display(messy.dropna())
display(messy.fillna({"age": messy["age"].median(),
                      "score": messy["score"].mean(),
                      "city": "Unknown"}))

Choose how to handle missing values based on context. Filling numeric data with a mean or median is not automatically correct; consider why values are missing and whether missingness itself is informative.

## 19. Duplicates and data types

In [ ]:
duplicates = pd.DataFrame({
    "id": [1, 2, 2, 3],
    "name": ["Ali", "Sara", "Sara", "Omar"]
})
print("Duplicate rows:", duplicates.duplicated().sum())
display(duplicates.drop_duplicates())
duplicates["id"] = duplicates["id"].astype("int64")

## 20. String operations

In [ ]:
people = pd.DataFrame({
    "name": ["  ALI Khan ", "Sara Ahmed", "omar malik", None],
    "email": ["ALI@EXAMPLE.COM", "sara@example.com", "omar@example.com", None]
})
people["name_clean"] = people["name"].str.strip().str.title()
people["email_clean"] = people["email"].str.lower()
people["has_example_email"] = people["email"].str.contains("example.com", na=False)
display(people)

## 21. GroupBy and aggregation

In [ ]:
sales = pd.DataFrame({
    "department": ["IT", "IT", "HR", "HR", "Sales", "Sales"],
    "employee": ["A", "B", "C", "D", "E", "F"],
    "salary": [40000, 50000, 35000, 37000, 42000, 46000],
    "years": [2, 5, 3, 4, 1, 6]
})
display(sales.groupby("department")["salary"].mean().sort_values(ascending=False))
display(sales.groupby("department").agg(
    average_salary=("salary", "mean"),
    maximum_salary=("salary", "max"),
    employee_count=("employee", "count"),
    average_experience=("years", "mean")
).reset_index())

## 22. Merge, join, and concatenate

In [ ]:
employees = pd.DataFrame({"employee_id": [1, 2, 3], "name": ["Ali", "Sara", "Omar"]})
payroll = pd.DataFrame({"employee_id": [1, 2, 4], "salary": [40000, 50000, 60000]})

display(employees.merge(payroll, on="employee_id", how="inner"))
display(employees.merge(payroll, on="employee_id", how="left"))
display(employees.merge(payroll, on="employee_id", how="outer", indicator=True))

north = pd.DataFrame({"city": ["Leeds"], "sales": [100]})
south = pd.DataFrame({"city": ["London"], "sales": [200]})
display(pd.concat([north, south], ignore_index=True))

**Join types:** `inner` keeps matching keys, `left` keeps all left rows, `right` keeps all right rows, and `outer` keeps all keys from both tables.

## 23. Dates and time series

In [ ]:
dates = pd.DataFrame({
    "date": ["2026-01-03", "2026-02-14", "not-a-date"],
    "sales": [100, 150, 80]
})
dates["date"] = pd.to_datetime(dates["date"], errors="coerce")
dates["year"] = dates["date"].dt.year
dates["month"] = dates["date"].dt.month
dates["weekday"] = dates["date"].dt.day_name()
display(dates)

### Resampling dated observations

In [ ]:
daily = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=10, freq="D"),
    "value": [10, 12, 9, 15, 11, 18, 17, 13, 16, 20]
}).set_index("date")

display(daily.resample("W")["value"].mean())

## 24. Pivot tables and crosstabs

In [ ]:
transactions = pd.DataFrame({
    "region": ["North", "North", "South", "South", "North", "South"],
    "product": ["A", "B", "A", "B", "A", "A"],
    "revenue": [100, 150, 120, 180, 90, 130]
})
display(pd.pivot_table(transactions, values="revenue", index="region",
                       columns="product", aggfunc="sum", fill_value=0))
display(pd.crosstab(transactions["region"], transactions["product"]))

## 25. Categorical data and memory

In [ ]:
levels = ["low", "medium", "high"]
priority = pd.Series(pd.Categorical(
    ["low", "high", "medium", "low"],
    categories=levels,
    ordered=True
))
print(priority.sort_values())
print("Memory usage (bytes):")
print(df.memory_usage(deep=True))

## 26. `apply`, `map`, and vectorised alternatives

In [ ]:
sample = pd.DataFrame({"price": [10, 20, 30, 40]})
sample["tax_vectorised"] = sample["price"] * 0.2
sample["tax_apply"] = sample["price"].apply(lambda value: value * 0.2)
display(sample)

Prefer vectorised operations such as `df["price"] * 0.2` when possible. `apply` is useful for some custom operations, but it is not automatically faster.

## 27. MultiIndex basics

In [ ]:
multi = sales.set_index(["department", "employee"]).sort_index()
display(multi)
print(multi.loc[("IT", "A")])
display(multi.reset_index())

## 28. Efficient data handling

- Select only the columns you need.
- Filter early when it reduces the data you must process.
- Use vectorised operations instead of Python loops where practical.
- Set suitable data types; categorical types can help for repeated labels.
- Read large files in chunks with `pd.read_csv(..., chunksize=...)`.
- Avoid chained assignment. Use `.loc[rows, columns] = value`.
- Profile before optimising; readability matters too.

In [ ]:
# Example: chunked CSV processing (uncomment and adapt to your file)
# total_rows = 0
# for chunk in pd.read_csv("large_file.csv", chunksize=100_000):
#     total_rows += len(chunk)
# print(total_rows)

## 29. Exporting data

In [ ]:
# Uncomment to write files into your current working directory:
# df.to_csv("students_clean.csv", index=False)
# df.to_excel("students_clean.xlsx", index=False)
# df.to_json("students_clean.json", orient="records", indent=2)

# Part III — Mini project: clean and analyse a dataset

In [ ]:
raw = pd.DataFrame({
    "student": ["Ali", "Sara", "Omar", "Mina", "Sara"],
    "course": ["Python", "Python", "Data Science", "Data Science", "Python"],
    "score": [78, 92, np.nan, 68, 92],
    "study_hours": [4.0, 7.5, 3.0, np.nan, 7.5],
    "city": ["London", "Leeds", "London", None, "Leeds"]
})

print("Raw data:")
display(raw)

clean = raw.drop_duplicates().copy()
clean["score"] = clean["score"].fillna(clean["score"].median())
clean["study_hours"] = clean["study_hours"].fillna(clean["study_hours"].median())
clean["city"] = clean["city"].fillna("Unknown")
clean["passed"] = clean["score"] >= 70

print("Clean data:")
display(clean)

print("Summary by course:")
display(clean.groupby("course").agg(
    students=("student", "count"),
    average_score=("score", "mean"),
    pass_rate=("passed", "mean"),
    average_study_hours=("study_hours", "mean")
).reset_index())

print("Correlation between study hours and score:")
print(clean[["study_hours", "score"]].corr())

**Think critically:** This is a teaching example, not a validated analysis. Median imputation and correlation can be misleading on very small datasets. Always examine the data and the problem before drawing conclusions.

## Pandas practice exercises

Try these yourself before checking the answer key:
1. Create a DataFrame with `name`, `age`, and `score`.
2. Select rows where score is at least 80.
3. Add a `passed` column where score is at least 50.
4. Group a sales table by category and calculate total sales.
5. Merge two DataFrames using a shared ID.
6. Convert a text column to lowercase and trim whitespace.
7. Parse a date column and extract its month.
8. Find and remove duplicate rows.
9. Fill missing numeric values with the median.
10. Create a pivot table showing total sales by region and product.

In [ ]:
# Write your solutions below.

## Pandas exercise answer key

In [ ]:
# 1
students = pd.DataFrame({
    "name": ["Ali", "Sara", "Omar"],
    "age": [22, 25, 23],
    "score": [75, 90, 48]
})
display(students)

# 2
display(students[students["score"] >= 80])

# 3
students["passed"] = students["score"] >= 50
display(students)

# 4
sales_example = pd.DataFrame({
    "category": ["A", "B", "A"],
    "sales": [100, 50, 75]
})
display(sales_example.groupby("category", as_index=False)["sales"].sum())

# 5
left = pd.DataFrame({"id": [1, 2], "name": ["Ali", "Sara"]})
right = pd.DataFrame({"id": [1, 2], "score": [70, 90]})
display(left.merge(right, on="id", how="inner"))

# 6
names = pd.Series(["  ALI ", "Sara  ", " omar"])
print(names.str.strip().str.lower())

# 7
dates = pd.to_datetime(pd.Series(["2026-01-05", "2026-03-20"]))
print(dates.dt.month)

# 8
duplicate_df = pd.DataFrame({"x": [1, 1, 2]})
print(duplicate_df.drop_duplicates())

# 9
missing_df = pd.DataFrame({"score": [10, np.nan, 30]})
missing_df["score"] = missing_df["score"].fillna(missing_df["score"].median())
display(missing_df)

# 10
pivot_example = pd.DataFrame({
    "region": ["North", "North", "South"],
    "product": ["A", "B", "A"],
    "sales": [10, 20, 30]
})
display(pd.pivot_table(
    pivot_example, values="sales", index="region",
    columns="product", aggfunc="sum", fill_value=0
))

## Study tip
Practise using `.loc` for label-based selection, `.iloc` for position-based selection, and `.copy()` before making an independent modified DataFrame.